# Adapter Evaluation

Generate lyrics from trained adapters, classify with RoBERTa, and compare against baselines.

In [1]:
import json

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams["pdf.fonttype"] = 42

from config import adapter_registry, Adapter, ARTISTS, RESULTS_DIR

# Display-only notebook: it reads cached results, no model is loaded here.
# Generate the cache first with:  uv run python evaluate.py

## Adapter results

Loaded from the cache written by `evaluate.py`. Run `uv run python evaluate.py` first.

In [2]:
# Load per-adapter results cached by evaluate.py (results/adapters/<name>.json).
# Keyed by display label; the Adapter object is re-attached from the registry.
# A STALE notice means the adapter was retrained after its cache was written.
CACHE_DIR = RESULTS_DIR / "adapters"


def _weights_mtime(path):
    return max(f.stat().st_mtime for f in path.rglob("*") if f.is_file())


all_results = {}

for a in adapter_registry():
    cache_file = CACHE_DIR / f"{a.name}.json"
    if not cache_file.exists():
        print(f"skip {a.label}: not evaluated -- run `uv run python evaluate.py`")
        continue

    hit = json.load(open(cache_file))
    if a.path.exists() and hit.get("mtime") != _weights_mtime(a.path):
        print(f"STALE {a.label}: weights newer than cache -- re-run evaluate.py")

    df = pd.DataFrame(hit["df"])
    all_results[a.label] = {"adapter": a, "target": a.artist,
                            "samples": hit["samples"], "df": df}
    print(f"loaded {a.label}: mean {df[a.artist].mean():.4f}")

skip Gojira LoRA r=8: not evaluated -- run `uv run python evaluate.py`
skip Gojira DoRA r=8: not evaluated -- run `uv run python evaluate.py`
skip Tool LoRA r=8: not evaluated -- run `uv run python evaluate.py`
skip Tool DoRA r=8: not evaluated -- run `uv run python evaluate.py`
skip Death LoRA r=8: not evaluated -- run `uv run python evaluate.py`
skip Death DoRA r=8: not evaluated -- run `uv run python evaluate.py`
skip Meshuggah LoRA r=8: not evaluated -- run `uv run python evaluate.py`
skip Meshuggah DoRA r=8: not evaluated -- run `uv run python evaluate.py`
skip Opeth LoRA r=8: not evaluated -- run `uv run python evaluate.py`
skip Opeth DoRA r=8: not evaluated -- run `uv run python evaluate.py`
skip Gojira LoRA r=4: not evaluated -- run `uv run python evaluate.py`
skip Gojira LoRA r=16: not evaluated -- run `uv run python evaluate.py`
skip Gojira LoRA r=8 SW: not evaluated -- run `uv run python evaluate.py`


## Baselines (from 04_baselines.ipynb)

Read `results/baselines.json` (written by `04`) so the numbers never drift from a hardcoded copy. Run `04` first.

In [ ]:
baselines = {}
baselines_path = RESULTS_DIR / "baselines.json"
if baselines_path.exists():
    with open(baselines_path) as f:
        baselines = json.load(f)   # {artist: {"zero_shot": {mean,std}, "few_shot": {mean,std}}}
else:
    print(f"note: {baselines_path} not found -- run 04_baselines first; showing adapters only\n")

rows = []
for artist, b in baselines.items():
    rows.append({"Method": f"{artist} Zero-shot", "Artist": artist,
                 "Target Attr. (mean)": b["zero_shot"]["mean"], "Target Attr. (std)": b["zero_shot"]["std"]})
    rows.append({"Method": f"{artist} Few-shot", "Artist": artist,
                 "Target Attr. (mean)": b["few_shot"]["mean"], "Target Attr. (std)": b["few_shot"]["std"]})

for label, data in all_results.items():
    target = data["target"]
    rows.append({"Method": label, "Artist": target,
                 "Target Attr. (mean)": data["df"][target].mean(),
                 "Target Attr. (std)": data["df"][target].std()})

summary = pd.DataFrame(rows)
summary["Target Attr. (mean)"] = summary["Target Attr. (mean)"].round(4)
summary["Target Attr. (std)"] = summary["Target Attr. (std)"].round(4)
print(summary.to_string(index=False))

## Plots

In [ ]:
# One panel per artist that appears in the summary (baseline or adapter).
artists_present = [a for a in ARTISTS if (summary["Artist"] == a).any()]
n = len(artists_present)
fig, axes = plt.subplots(1, n, figsize=(6 * n, 5), sharey=True)
if n == 1:
    axes = [axes]

for ax, artist in zip(axes, artists_present):
    artist_rows = summary[summary["Artist"] == artist].copy()
    artist_rows["short"] = artist_rows["Method"].str.replace(f"{artist} ", "", regex=False)
    means = artist_rows["Target Attr. (mean)"].values
    stds = artist_rows["Target Attr. (std)"].values
    labels_plot = artist_rows["short"].values
    x = np.arange(len(labels_plot))

    colors = []
    for label in labels_plot:
        if "Zero" in label:
            colors.append("#9e9e9e")
        elif "Few" in label:
            colors.append("#607d8b")
        elif "SW" in label:
            colors.append("#2e7d32")
        elif "DoRA" in label:
            colors.append("#e65100")
        else:
            colors.append("#1565c0")

    bars = ax.bar(x, means, yerr=stds, capsize=4, color=colors, edgecolor="black", linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(labels_plot, rotation=30, ha="right", fontsize=9)
    ax.set_title(f"{artist}", fontsize=13, fontweight="bold")
    ax.set_ylim(0, 1.15)
    ax.axhline(y=1.0, color="gray", linestyle="--", linewidth=0.5, alpha=0.5)

    for bar, m in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.03,
                f"{m:.2f}", ha="center", va="bottom", fontsize=8)

axes[0].set_ylabel("Target-Artist Attribution")
plt.suptitle("Target-Artist Attribution by Method", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../report/figures/method_comparison.pdf", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

# LoRA vs DoRA: every artist that has BOTH a lora r8 and dora r8 result.
lora_dora_artists = [
    a for a in ARTISTS
    if Adapter(a, "lora", 8).label in all_results and Adapter(a, "dora", 8).label in all_results
]

ax = axes[0]
x = np.arange(len(lora_dora_artists))
w = 0.35
for i, (kind, color) in enumerate([("lora", "#1565c0"), ("dora", "#e65100")]):
    means, stds = [], []
    for artist in lora_dora_artists:
        d = all_results[Adapter(artist, kind, 8).label]
        means.append(d["df"][d["target"]].mean())
        stds.append(d["df"][d["target"]].std())
    ax.bar(x + i * w, means, w, yerr=stds, capsize=4, label=kind.upper(),
           color=color, edgecolor="black", linewidth=0.5)
ax.set_xticks(x + w / 2)
ax.set_xticklabels(lora_dora_artists, rotation=20, ha="right")
ax.set_ylabel("Target-Artist Attribution")
ax.set_title("LoRA vs DoRA (r=8)", fontweight="bold")
ax.legend()
ax.set_ylim(0, 1.15)

# Rank ablation (Gojira): whichever of r=4/8/16 are present.
ax = axes[1]
rank_specs = [s for s in (Adapter("Gojira", "lora", r) for r in (4, 8, 16)) if s.label in all_results]
rank_labels = [f"r={s.rank}" for s in rank_specs]
rank_means = [all_results[s.label]["df"]["Gojira"].mean() for s in rank_specs]
rank_stds = [all_results[s.label]["df"]["Gojira"].std() for s in rank_specs]
bars = ax.bar(rank_labels, rank_means, yerr=rank_stds, capsize=4,
              color=["#42a5f5", "#1565c0", "#0d47a1"][:len(rank_specs)],
              edgecolor="black", linewidth=0.5)
for bar, m in zip(bars, rank_means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.03,
            f"{m:.2f}", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Gojira Attribution")
ax.set_xlabel("LoRA Rank")
ax.set_title("Rank Ablation (Gojira)", fontweight="bold")
ax.set_ylim(0, 1.15)

plt.tight_layout()
plt.savefig("../report/figures/ablation.pdf", bbox_inches="tight", dpi=300)
plt.show()

## Sample Lyrics

Best sample per adapter (highest target-artist attribution).

In [7]:
for name, data in all_results.items():
    target = data["target"]
    best_idx = data["df"][target].idxmax()
    score = data["df"].loc[best_idx, target]
    text = data["samples"][best_idx]
    print(f"{'='*60}")
    print(f"{name} — best sample (attribution: {score:.3f})")
    print(f"{'='*60}")
    print(text[:800])
    print("...\n" if len(text) > 800 else "\n")

Gojira LoRA r=8 — best sample (attribution: 0.991)
I don't remember when I started to doubt myself
The answers are coming from my mind, all around me
It's no use complaining 'bout the way I have been ruled
My own freedom is in the hands of none but me
I used to be so proud and pretentious
But now I see through everything
There has always been a reason for what we do
That's the only truth that matters


On my side stands my secret weapon
All around lies fear and confusion
And as I grow older I start to understand
This world ain't real at all


Now I find shelter under my wings, I rest on my arms
Surrender to this great power inside, let it flow through me
I open myself wide to this world, awake my soul
Open your eyes now
It comes to life, I hear the roar
Of giants living deep inside of us
With every step you take, you learn something new
Ab
...

Gojira DoRA r=8 — best sample (attribution: 0.991)
For those of you that think we're all alone
We are not, our soul is awake
We take back what 